The calendar is generated for the full years covered by the fact instead of
being built from the 13 distinct dates present in the file, so it stays
valid when new extracts arrive.

In [0]:
from pyspark.sql import functions as F

In [0]:
SOURCE_CATALOG_NAME = 'beverage_sales'
SOURCE_SCHEMA_NAME = 'silver'
SOURCE_TABLE_NAME = 'sales'

TARGET_CATALOG_NAME = 'beverage_sales'
TARGET_SCHEMA_NAME = 'gold'
TARGET_TABLE_NAME = 'dim_date'

In [0]:

bounds = (
    spark.table(f'{SOURCE_CATALOG_NAME}.{SOURCE_SCHEMA_NAME}.{SOURCE_TABLE_NAME}')
    .agg(
        F.min('sales_date').alias('min_date'),
        F.max('sales_date').alias('max_date')
    )
    .first()
)

In [0]:
df_dim_date = (
    spark.range(1)
    .select(
        F.explode(
            F.sequence(
                F.trunc(F.lit(bounds['min_date']), 'YEAR'),
                F.add_months(F.trunc(F.lit(bounds['max_date']), 'YEAR'), 12) - 1,
                F.expr('INTERVAL 1 DAY')
            )
        ).alias('date')
    )
    .withColumn('date_key', F.date_format('date', 'yyyyMMdd').cast('int'))
    .withColumn('year', F.year('date'))
    .withColumn('month', F.month('date'))
    .withColumn('month_name', F.date_format('date', 'MMMM'))
    .withColumn('year_month', F.date_format('date', 'yyyy-MM'))
    .withColumn('quarter', F.quarter('date'))
    .withColumn('period', F.ceil(F.dayofmonth('date') / 7).cast('int'))
    .withColumn('day_name', F.date_format('date', 'EEEE'))
    .withColumn('is_weekend', F.dayofweek('date').isin(1, 7))
    .select(
        'date_key',
        'date',
        'year',
        'month',
        'month_name',
        'year_month',
        'quarter',
        'period',
        'day_name',
        'is_weekend'
    )
)

In [0]:
df_dim_date\
    .write\
    .mode('overwrite')\
    .saveAsTable(f'{TARGET_CATALOG_NAME}.{TARGET_SCHEMA_NAME}.{TARGET_TABLE_NAME}')